# Round 7: scale the averaging win, ExtraTrees, family blend - S6E10

Round 6 produced the first working-signal winner over the incumbent: Exp 22, a 3-seed average of a
lightly stochastic LightGBM (+0.00031 mean paired delta, all five folds positive; descriptive, not
proof). This round follows that evidence with three bounded tests, still on existing data only:

- **Exp 23: 7-seed averaging** (seeds 42-48) of the same stochastic config. If the Exp 22 gain is
  variance reduction, more seeds should hold or grow it with diminishing returns; if it was luck,
  it should fade. The in-run 3-seed subset (42/43/44) doubles as a cross-run determinism check
  against round 6's Exp 22 folds.
- **Exp 24: ExtraTrees** (Andrea's preferred forest family, the untested variant): 400 trees,
  min_samples_leaf=5 - the one knob that genuinely improved RF over its default. ExtraTrees splits
  more randomly than RF, which often decorrelates it from gradient boosting - exactly what a blend
  needs. Measured, not assumed.
- **Exp 25: fixed 50/50 blend** of Exp 23 + Exp 24 OOF probabilities. The weight is fixed a priori
  (no scan - a weight chosen on the same OOF that scores it would be selection). The OOF correlation
  between the two families is reported as the complementarity diagnostic: blending only makes sense
  if the errors decorrelate.

Reading rules unchanged: paired per-fold deltas, WORKING SIGNAL / NO SIGNAL under the 2x band -
a decision heuristic, not an inferential test. Incumbent recomputed in-run as always.


In [ ]:
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split

RANDOM_STATE = 42
N_SPLITS = 5
CATEGORICAL = ['Gender', 'Customer Type', 'Type of Travel', 'Class']
INCUMBENT_FOLDS_R3 = [0.95851, 0.95751, 0.95907, 0.95841, 0.95885]  # determinism reference
E22_FOLDS_R6 = [0.95886, 0.95756, 0.95953, 0.95875, 0.95922]        # round-6 Exp 22 (3-seed avg) folds
STOCH = dict(bagging_fraction=0.9, bagging_freq=1, feature_fraction=0.9)


In [ ]:
competition_dir = Path('.')
if not (competition_dir / 'data' / 'train.csv').is_file() and Path('playground-series-s6e10/data/train.csv').is_file():
    competition_dir = Path('playground-series-s6e10')
data_dir = competition_dir / 'data'

train = pd.read_csv(data_dir / 'train.csv')

def to_binary(series):
    if series.dtype == bool:
        return series.astype(int)
    return series.map({'True': 1, 'False': 0, True: 1, False: 0}).astype(int)

target = to_binary(train['satisfaction'])
features = train.drop(columns=['id', 'satisfaction'])

def to_lgbm(df):
    out = df.copy()
    for col in CATEGORICAL:
        out[col] = out[col].astype('category')
    return out

def to_sklearn(train_part, other_parts=()):
    # sklearn forests reject NaN: median-impute numerics (fit on train_part only),
    # one-hot the four categoricals (dummy column set fixed by train_part).
    num_cols = [c for c in train_part.columns if c not in CATEGORICAL]
    medians = train_part[num_cols].median()
    def prep(df):
        out = df.copy()
        out[num_cols] = out[num_cols].fillna(medians)
        return pd.get_dummies(out, columns=CATEGORICAL)
    ref = prep(train_part)
    cols = ref.columns
    return ref, [prep(p).reindex(columns=cols, fill_value=0) for p in other_parts]

def oof_lgbm(feat_frame, label, name, seed=RANDOM_STATE, **params):
    frame = to_lgbm(feat_frame)
    oof = np.zeros(len(frame))
    aucs, iters = [], []
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)  # same folds every time
    for fold, (tr_idx, va_idx) in enumerate(skf.split(frame, label), 1):
        X_tr, y_tr = frame.iloc[tr_idx], label.iloc[tr_idx]
        X_va, y_va = frame.iloc[va_idx], label.iloc[va_idx]
        X_fit, X_es, y_fit, y_es = train_test_split(
            X_tr, y_tr, test_size=0.1, stratify=y_tr, random_state=RANDOM_STATE)
        # caller params merge over defaults (round-6 disclosed fix - never pass a key twice)
        cfg = dict(learning_rate=0.05, num_leaves=63, n_estimators=1000,
                   n_jobs=-1, random_state=seed, verbose=-1)
        cfg.update(params)
        model = lgb.LGBMClassifier(**cfg)
        model.fit(X_fit, y_fit, eval_set=[(X_es, y_es)], eval_metric='auc',
                  callbacks=[lgb.early_stopping(50, verbose=False)])
        oof[va_idx] = model.predict_proba(X_va)[:, 1]
        a = roc_auc_score(y_va, oof[va_idx])
        aucs.append(a); iters.append(model.best_iteration_)
        print(f'  {name} fold {fold}: AUC {a:.5f} (best_iteration={model.best_iteration_})')
    return oof, aucs, iters

def fold_aucs(oof, label):
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    return [roc_auc_score(label.iloc[va], oof[va]) for _, va in skf.split(oof.reshape(-1, 1), label)]

print(f'Full train: {len(features):,} rows, {features.shape[1]} raw features')


## Reference: incumbent recompute

Same-run incumbent (Exp 8 config) with the determinism check against the round-3 constants, as every
round since round 5.


In [ ]:
print('Reference: incumbent raw LightGBM (Exp 8 config), recomputed in-run')
oof_inc, aucs_inc, iters_inc = oof_lgbm(features, target, 'INC')
diffs = [abs(a - b) for a, b in zip(aucs_inc, INCUMBENT_FOLDS_R3)]
det_note = 'PASS - identical folds' if max(diffs) < 1e-4 else 'WARN - use in-run folds only'
print(f'  determinism vs round-3 constants: max abs diff {max(diffs):.5f} ({det_note})')
INC_FOLDS = aucs_inc
INC_MEAN = float(np.mean(aucs_inc))
print(f'  incumbent mean OOF AUC: {INC_MEAN:.5f} +/- {np.std(aucs_inc):.5f} | pooled: {roc_auc_score(target, oof_inc):.6f}')


## Exp 23: 7-seed averaging of the stochastic config

Seeds 42-48 of the Exp 21 config, OOF probabilities averaged per row. The seeds-42/43/44 subset is
scored too: it must reproduce round 6's Exp 22 folds (cross-run determinism of the stochastic
config), and it is the paired comparator for the 4 extra seeds.


In [ ]:
print('Exp 23: 7-seed average of the stochastic LightGBM config')
SEEDS = [42, 43, 44, 45, 46, 47, 48]
oof_seeds = {}
iters42 = None
for seed in SEEDS:
    o, a, it = oof_lgbm(features, target, f'E23-s{seed}', seed=seed, **STOCH)
    oof_seeds[seed] = o
    if seed == 42:
        iters42 = it  # kept for the refit best-iteration basis
    print(f'  seed {seed} mean AUC: {np.mean(a):.5f}')
oof23 = np.mean([oof_seeds[s] for s in SEEDS], axis=0)
aucs23 = fold_aucs(oof23, target)
for fold, a in enumerate(aucs23, 1):
    print(f'  E23 fold {fold} (7-seed avg): AUC {a:.5f}')
print(f'Exp 23 mean OOF AUC: {np.mean(aucs23):.5f} +/- {np.std(aucs23):.5f} | pooled: {roc_auc_score(target, oof23):.6f}')

oof22r = np.mean([oof_seeds[s] for s in [42, 43, 44]], axis=0)  # in-run Exp 22 reproduction
aucs22r = fold_aucs(oof22r, target)
diffs = [abs(a - b) for a, b in zip(aucs22r, E22_FOLDS_R6)]
det22 = 'PASS' if max(diffs) < 1e-4 else 'WARN'
print(f'  in-run 3-seed subset folds: ' + ' '.join(f'{a:.5f}' for a in aucs22r))
print(f'  cross-run determinism vs round-6 Exp 22: max abs diff {max(diffs):.5f} ({det22})')


## Exp 24: ExtraTrees 400, min_samples_leaf=5

The untested forest variant, honoring the RF preference with a measured case: leaf=5 was the one
knob that improved RF over its default, and ExtraTrees' random thresholds often decorrelate its
errors from gradient boosting. sklearn forests reject NaN, so numerics are median-imputed per fold
(fit on the fold's training rows only) and categoricals one-hot encoded - the same representation
family as the earlier RF experiments.


In [ ]:
print('Exp 24: ExtraTreesClassifier n_estimators=400, min_samples_leaf=5, one-hot')
oof24 = np.zeros(len(features))
aucs24 = []
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
for fold, (tr_idx, va_idx) in enumerate(skf.split(features, target), 1):
    X_tr, y_tr = features.iloc[tr_idx], target.iloc[tr_idx]
    X_va, y_va = features.iloc[va_idx], target.iloc[va_idx]
    X_tr_p, (X_va_p,) = to_sklearn(X_tr, [X_va])
    et = ExtraTreesClassifier(n_estimators=400, min_samples_leaf=5, n_jobs=-1,
                              random_state=RANDOM_STATE)
    et.fit(X_tr_p, y_tr)
    oof24[va_idx] = et.predict_proba(X_va_p)[:, 1]
    a = roc_auc_score(y_va, oof24[va_idx])
    aucs24.append(a)
    print(f'  E24 fold {fold}: AUC {a:.5f}')
print(f'Exp 24 mean OOF AUC: {np.mean(aucs24):.5f} +/- {np.std(aucs24):.5f} | pooled: {roc_auc_score(target, oof24):.6f}')


## Exp 25: fixed 50/50 blend of Exp 23 + Exp 24

Weight fixed a priori at 0.5/0.5 - no weight scan, because a weight chosen on the same OOF that
scores it would be selection on the evaluated set. The Pearson correlation between the two families'
OOF probabilities is reported first: it is the measured evidence for or against complementarity.


In [ ]:
corr = float(np.corrcoef(oof23, oof24)[0, 1])
print(f'OOF probability correlation, Exp 23 (LGBM 7-seed) vs Exp 24 (ExtraTrees): {corr:.4f}')
print('  (high correlation is expected - both model the same target; the blend question is whether')
print('   the residual disagreement ranks the validation rows any better)')
oof25 = 0.5 * oof23 + 0.5 * oof24
aucs25 = fold_aucs(oof25, target)
for fold, a in enumerate(aucs25, 1):
    print(f'  E25 fold {fold} (blend): AUC {a:.5f}')
print(f'Exp 25 mean OOF AUC: {np.mean(aucs25):.5f} +/- {np.std(aucs25):.5f} | pooled: {roc_auc_score(target, oof25):.6f}')


## Results, verdicts, and refit guard

Paired per-fold deltas: Exp 23 vs the in-run incumbent AND vs the in-run 3-seed subset (the marginal
value of 4 more seeds); Exp 24 and Exp 25 vs the incumbent; Exp 25 also vs Exp 23 alone (does the
forest add anything to the averaged booster?). Whole-data winners with a working signal earn a
refit; a no-signal round writes no submission file.


In [ ]:
def paired_verdict(new_aucs, ref_aucs):
    deltas = [n - r for n, r in zip(new_aucs, ref_aucs)]
    mean_d, spread = float(np.mean(deltas)), float(np.std(deltas))
    if abs(mean_d) <= 2 * spread:
        label = 'NO SIGNAL (within 2x per-fold delta spread - no action)'
    else:
        label = ('WORKING SIGNAL: better' if mean_d > 0 else 'WORKING SIGNAL: worse') + ' (descriptive, not proof)'
    return deltas, mean_d, spread, label

results = {}
for exp_id, name, aucs, oof, extra_refs in [
    ('23', 'LGBM stochastic 7-seed avg', aucs23, oof23, [('in-run 3-seed subset', aucs22r)]),
    ('24', 'ExtraTrees 400/leaf5', aucs24, oof24, []),
    ('25', 'blend 50/50 E23+E24', aucs25, oof25, [('Exp 23', aucs23)]),
]:
    deltas, mean_d, spread, label = paired_verdict(aucs, INC_FOLDS)
    results[exp_id] = dict(name=name, aucs=aucs, oof=oof, mean_d=mean_d, label=label)
    print(f'Exp {exp_id} {name}: mean {np.mean(aucs):.5f} +/- {np.std(aucs):.5f} | pooled {roc_auc_score(target, oof):.6f}')
    print(f'  paired deltas vs in-run incumbent: ' + ' '.join(f'{x:+.5f}' for x in deltas) + f' (mean {mean_d:+.5f})')
    print(f'  {label}')
    for ref_name, ref_aucs in extra_refs:
        d2, m2, s2, lab2 = paired_verdict(aucs, ref_aucs)
        print(f'  paired vs {ref_name}: mean {m2:+.5f} | {lab2}')
print()

best_oof = max([INC_MEAN] + [float(np.mean(r['aucs'])) for r in results.values()])
print(f'Best OOF mean incl. incumbent: {best_oof:.5f} - >=0.96: {best_oof >= 0.96} (descriptive; target claims stay cautious - repeated selection on this data)')

winners = [k for k, r in results.items()
           if r['mean_d'] > 0 and 'WORKING SIGNAL: better' in r['label']]
if not winners:
    print('No candidate earned a refit: incumbent Exp 2 raw LightGBM stands; submission_round6.csv (Exp 22 refit) remains the most recent submission candidate.')
else:
    win = max(winners, key=lambda k: np.mean(results[k]['aucs']))
    print(f'Winner: Exp {win} ({results[win]["name"]}) - refits on all rows below; logged as a working signal, exploratory.')


In [ ]:
# Refit ONLY fires on a whole-data working-signal winner over the standing incumbent.
if not winners:
    print('No submission file written this round.')
else:
    test = pd.read_csv(data_dir / 'test.csv')
    test_features = test.drop(columns=['id'])
    def full_stochastic(seed, n_est):
        m = lgb.LGBMClassifier(learning_rate=0.05, num_leaves=63, n_estimators=n_est,
                               n_jobs=-1, random_state=seed, verbose=-1, **STOCH)
        m.fit(to_lgbm(features), target)
        return m.predict_proba(to_lgbm(test_features))[:, 1]
    # best-iteration basis: mean of THIS run's seed-42 stochastic fold best_iterations (iters42) -
    # the same semantics round 6 logged as 313; seed-shared across all seven refit seeds by design.
    N_EST_STOCH = int(np.mean(iters42))
    print(f'  refit n_estimators basis (seed-42 fold mean best_iteration): {N_EST_STOCH}')
    if win == '23':
        test_probabilities = np.mean([full_stochastic(s, N_EST_STOCH) for s in SEEDS], axis=0)
    elif win == '24':
        X_all, (X_test_p,) = to_sklearn(features, [test_features])
        et = ExtraTreesClassifier(n_estimators=400, min_samples_leaf=5, n_jobs=-1,
                                  random_state=RANDOM_STATE)
        et.fit(X_all, target)
        test_probabilities = et.predict_proba(X_test_p)[:, 1]
    else:  # '25': 50/50 blend of the two full-data pipelines
        p_lgbm = np.mean([full_stochastic(s, N_EST_STOCH) for s in SEEDS], axis=0)
        X_all, (X_test_p,) = to_sklearn(features, [test_features])
        et = ExtraTreesClassifier(n_estimators=400, min_samples_leaf=5, n_jobs=-1,
                                  random_state=RANDOM_STATE)
        et.fit(X_all, target)
        test_probabilities = 0.5 * p_lgbm + 0.5 * et.predict_proba(X_test_p)[:, 1]
    submission = pd.read_csv(data_dir / 'sample_submission.csv')
    submission['satisfaction'] = test_probabilities
    assert len(submission) == len(test)
    assert submission['id'].equals(test['id'])
    out_path = competition_dir / 'submission_round7.csv'
    submission.to_csv(out_path, index=False)
    print(f"Saved {len(submission):,} predictions to {out_path} - submitting stays Andrea's call.")


In [ ]:
# OOF cache (established pattern): persist predictions + metadata for later rounds.
import hashlib, json as _json, time

cache_dir = competition_dir / 'oof_cache'
cache_dir.mkdir(exist_ok=True)
h = hashlib.sha256()
with open(competition_dir / 'data' / 'train.csv', 'rb') as f:
    for chunk in iter(lambda: f.read(1 << 20), b''):
        h.update(chunk)
np.savez(cache_dir / 'round7_oof.npz',
         oof_inc=np.asarray(oof_inc), aucs_inc=np.asarray(aucs_inc), iters_inc=np.asarray(iters_inc),
         oof23=np.asarray(oof23), aucs23=np.asarray(aucs23),
         oof22r=np.asarray(oof22r), aucs22r=np.asarray(aucs22r),
         oof24=np.asarray(oof24), aucs24=np.asarray(aucs24),
         oof25=np.asarray(oof25), aucs25=np.asarray(aucs25),
         y=np.asarray(target).astype(int))
meta = {
    'round': 7,
    'protocol': '5-fold stratified OOF, seed 42; LightGBM inner-10% early stop per fold; ExtraTrees median-impute + one-hot per fold',
    'dataset': {'file': 'data/train.csv', 'rows': int(len(target)), 'sha256': h.hexdigest()},
    'configs': {
        'incumbent': 'LightGBM lr=0.05, num_leaves=63, n_estimators=1000, early stop 50 (Exp 8 config, recomputed in-run)',
        'exp23': 'LightGBM lr=0.05, num_leaves=63, bagging_fraction=0.9, bagging_freq=1, feature_fraction=0.9; seeds 42-48, OOF probabilities averaged',
        'exp24': 'ExtraTreesClassifier n_estimators=400, min_samples_leaf=5, random_state=42; per-fold median impute + one-hot',
        'exp25': '0.5*exp23 + 0.5*exp24 on OOF probabilities (weight fixed a priori)',
    },
    'saved_at': time.strftime('%Y-%m-%d %H:%M:%S %Z'),
}
(cache_dir / 'round7_oof.json').write_text(_json.dumps(meta, indent=2))
print('OOF cache written to', cache_dir)


## What happens next

- If Exp 23 holds or grows the Exp 22 gain: seed averaging is a real, cheap lever on this data -
  the natural follow-ups are seed x config mixes (average across the near-tied plateau configs, not
  just seeds) and checking whether the 3-seed submission candidate should be upgraded to 7.
- If Exp 24 lands near the forests' usual ~0.956-0.957 but Exp 25 still wins: evidence the families
  complement - the blend becomes the reference and weighted blending (scored on a fresh
  re-partition, never on the selection OOF) is the measured next step.
- If everything is null: the on-data evidence then spans capacity, representation, features,
  regularization, stochasticity, averaging, segments, and forests - the approaches tested so far
  have not reached 0.96 on this data alone (a finite set of configs, not proof it cannot be done),
  and Andrea's options stay as logged: external original-survey data (needs his explicit OK),
  heavier ensembling, or accepting a strong sub-0.96 model.
